# 🎙️ Proyecto Auditoria DIGI — Fase 1: Procesado de Audio

**Nodos cubiertos:** 1. ffmpeg → 2. Reducción de ruido → 3. Silero VAD → 4. VoxLingua107 (LID)

> **Nota Colab vs Docker:** En Colab usamos `noisereduce` (instalación inmediata, sin reinicio). En el Docker final se usa **DeepFilterNet 3** que da mejor calidad. La lógica del pipeline es idéntica — solo se intercambia el modelo de ruido.

---
**Orden de ejecución:** ejecuta las celdas de arriba a abajo, una a una.

## ✅ Paso 0 — Verificar GPU

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    print('✅ GPU disponible:')
    for linea in r.stdout.split('\n'):
        if any(x in linea for x in ['Tesla','RTX','A100','T4','V100','L4']):
            print(' ', linea.strip()); break
else:
    print('❌ Sin GPU — ve a: Entorno de ejecución → Cambiar tipo → GPU')

## 📦 Paso 1 — Instalar dependencias
Tarda ~1 minuto. No requiere reinicio.

In [ ]:
# Reducción de ruido (Colab: noisereduce | Docker: DeepFilterNet 3)
!pip install -q noisereduce

# Silero VAD — detección de voz
!pip install -q silero-vad

# SpeechBrain — identificación de idioma
!pip install -q speechbrain

# Audio utils
!pip install -q torchaudio soundfile

print('\n✅ Todo instalado — puedes continuar')

## 🎵 Paso 2 — Subir archivo de audio
Sube un .ogg, .mp3 o .m4a de prueba.

In [ ]:
import os
from pathlib import Path
from google.colab import drive

# ── Montar Drive ──────────────────────────────────────────────────
drive.mount('/content/drive')

# ── Ruta base del proyecto en tu Drive ───────────────────────────
# Ajusta esta variable con la carpeta donde tienes el proyecto
BASE_DIR = Path('/content/drive/MyDrive/auditoria-digi')
INPUT_DIR  = BASE_DIR / 'inputs'
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Archivo de audio de entrada ───────────────────────────────────
# Pon el nombre del archivo de llamada dentro de inputs/
NOMBRE_AUDIO = 'llamada.ogg'   # ← cambia al nombre de tu archivo
ARCHIVO_ORIGINAL = INPUT_DIR / NOMBRE_AUDIO

if ARCHIVO_ORIGINAL.exists():
    size_mb = ARCHIVO_ORIGINAL.stat().st_size / 1024 / 1024
    print(f'✅ Audio encontrado: {ARCHIVO_ORIGINAL}')
    print(f'   Tamaño: {size_mb:.1f} MB')
else:
    print(f'❌ No se encontró: {ARCHIVO_ORIGINAL}')
    print(f'   Sube tu archivo de llamada a Drive → {INPUT_DIR}/')
    raise FileNotFoundError(f'{ARCHIVO_ORIGINAL}')

ARCHIVO_ORIGINAL = str(ARCHIVO_ORIGINAL)

## 🔧 Nodo 1 — ffmpeg: Normalizar audio
Convierte cualquier formato a WAV mono 16kHz. Estándar que necesitan todos los modelos de voz.

In [ ]:
import subprocess, os

ARCHIVO_WAV = 'audio_normalizado.wav'

cmd = ['ffmpeg', '-y', '-i', ARCHIVO_ORIGINAL,
       '-ac', '1', '-ar', '16000', '-af', 'loudnorm', ARCHIVO_WAV]

r = subprocess.run(cmd, capture_output=True, text=True)

if r.returncode == 0:
    print(f'✅ Nodo 1 completado')
    print(f'   {os.path.getsize(ARCHIVO_ORIGINAL)/1024:.1f} KB → {os.path.getsize(ARCHIVO_WAV)/1024:.1f} KB')
    print(f'   Formato: mono, 16kHz, volumen normalizado')
else:
    print(f'❌ Error:\n{r.stderr[-400:]}')

## 🔇 Nodo 2 — Reducción de ruido
Elimina ruido de fondo, eco y estática de telefonía para que Whisper transcriba mejor.

> *Colab: noisereduce — Docker final: DeepFilterNet 3*

In [ ]:
import numpy as np
import soundfile as sf
import noisereduce as nr

ARCHIVO_LIMPIO = 'audio_sin_ruido.wav'

print('🔇 Aplicando reducción de ruido...')
audio, sr = sf.read(ARCHIVO_WAV)

# Reducir ruido — usa los primeros 0.5s como muestra de ruido de fondo
muestra_ruido = audio[:int(sr * 0.5)]
audio_limpio = nr.reduce_noise(y=audio, sr=sr, y_noise=muestra_ruido, prop_decrease=0.8)

sf.write(ARCHIVO_LIMPIO, audio_limpio, sr)

print(f'✅ Nodo 2 completado — ruido reducido')
print(f'   Archivo limpio: {os.path.getsize(ARCHIVO_LIMPIO)/1024:.1f} KB')

## ✂️ Nodo 3 — Silero VAD: Recortar silencios
Detecta cuándo hay voz. Elimina silencios para que Whisper no pierda tiempo ni invente palabras.

In [ ]:
import torch, torchaudio
from silero_vad import load_silero_vad, read_audio, get_speech_timestamps, collect_chunks

ARCHIVO_VAD = 'audio_vad.wav'
SAMPLE_RATE = 16000

print('✂️ Cargando Silero VAD v5...')
model_vad = load_silero_vad()

wav = read_audio(ARCHIVO_LIMPIO, sampling_rate=SAMPLE_RATE)

timestamps = get_speech_timestamps(
    wav, model_vad, sampling_rate=SAMPLE_RATE,
    threshold=0.5,
    min_speech_duration_ms=250,
    min_silence_duration_ms=300,
)

audio_vad = collect_chunks(timestamps, wav)
torchaudio.save(ARCHIVO_VAD, audio_vad.unsqueeze(0), SAMPLE_RATE)

duracion_original = len(wav) / SAMPLE_RATE
duracion_vad = len(audio_vad) / SAMPLE_RATE
reduccion = (1 - duracion_vad / duracion_original) * 100

del model_vad

print(f'✅ Nodo 3 completado')
print(f'   Fragmentos de voz detectados: {len(timestamps)}')
print(f'   {duracion_original:.1f}s → {duracion_vad:.1f}s  (silencios eliminados: {reduccion:.1f}%)')

## 🌍 Nodo 4 — VoxLingua107: Identificar idioma
Detecta el idioma de la llamada. Si no es español, genera un flag de alerta.

In [ ]:
import torchaudio, torch
from speechbrain.pretrained import EncoderClassifier

IDIOMA_ESPERADO = 'es'

print('🌍 Cargando VoxLingua107...')
classifier = EncoderClassifier.from_hparams(
    source='speechbrain/lang-id-voxlingua107-ecapa',
    savedir='tmp_lid'
)

signal, _ = torchaudio.load(ARCHIVO_VAD)
prediction = classifier.classify_batch(signal)
idioma_detectado = prediction[3][0].split(':')[0].strip()
confianza = float(prediction[1][0].max().exp()) * 100
flag_idioma = idioma_detectado != IDIOMA_ESPERADO

del classifier
torch.cuda.empty_cache()

print(f'\n✅ Nodo 4 completado')
print(f'   Idioma detectado: {idioma_detectado} ({confianza:.1f}% confianza)')
print(f'   {"⚠️ FLAG: idioma incorrecto para el territorio" if flag_idioma else "✅ Idioma correcto"}')

## 📋 Resumen — Fase 1 completada

In [ ]:
import json

resultado_fase1 = {
    'archivo_original': ARCHIVO_ORIGINAL,
    'archivo_salida': ARCHIVO_VAD,
    'duracion_original_s': round(duracion_original, 2),
    'duracion_procesada_s': round(duracion_vad, 2),
    'reduccion_silencios_pct': round(reduccion, 1),
    'idioma_detectado': idioma_detectado,
    'idioma_esperado': IDIOMA_ESPERADO,
    'confianza_idioma_pct': round(confianza, 1),
    'flags': {'idioma_incorrecto': flag_idioma}
}

print('=' * 55)
print('  FASE 1 — RESULTADO')
print('=' * 55)
for k, v in resultado_fase1.items():
    if k != 'flags':
        print(f'  {k}: {v}')
print()
print('  FLAGS:')
for flag, valor in resultado_fase1['flags'].items():
    print(f'    {flag}: {"⚠️ ACTIVO" if valor else "✅ OK"}')
print('=' * 55)
print(f'\n➡️  Audio listo para Fase 2: {ARCHIVO_VAD}')

with open('resultado_fase1.json', 'w') as f:
    json.dump(resultado_fase1, f, indent=2, ensure_ascii=False)
print('   Metadatos guardados: resultado_fase1.json')